## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:曹文静


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <vector>
#include <algorithm>
#include <cstdio>
#include <cassert>
using namespace std;

typedef long long LL;

// 常量定义
const int MAX_ARRAY_SIZE = 1024;
const int INPUT_BUFFER_SIZE = 1 << 17;

// 全局变量：数组、位置映射、参数、操作记录
int data_arr[MAX_ARRAY_SIZE];
int pos_map[MAX_ARRAY_SIZE];
int total_size, target_a, target_b, block_size;
vector<int> operation_list;

// 快速读入：字符级缓冲
inline char get_char() {
    static char buffer[INPUT_BUFFER_SIZE];
    static char *head = buffer, *tail = buffer;
    if (head == tail) {
        tail = buffer + fread(buffer, 1, INPUT_BUFFER_SIZE, stdin);
        head = buffer;
        if (head == tail) return EOF;
    }
    return *head++;
}

// 快速读入整数
inline int read_integer() {
    int result = 0;
    char ch = get_char();
    while (ch < '0') ch = get_char();
    while (ch >= '0') {
        result = result * 10 + (ch - '0');
        ch = get_char();
    }
    return result;
}

// 更新元素位置映射表
void update_position_map() {
    for (int i = 0; i < total_size; ++i) {
        pos_map[data_arr[i]] = i;
    }
}

// 交换target_a和target_b
void magic_swap() {
    operation_list.push_back(0);
    for (int i = 0; i < total_size; ++i) {
        if (data_arr[i] == target_a) {
            data_arr[i] = target_b;
        } else if (data_arr[i] == target_b) {
            data_arr[i] = target_a;
        }
    }
    update_position_map();
}

// 加法操作：每个元素加x模总大小
void add_operation(int x) {
    if (x == 0) return;
    operation_list.push_back(x);
    for (int i = 0; i < total_size; ++i) {
        data_arr[i] = (data_arr[i] + x) % total_size;
    }
    update_position_map();
}

// 异或操作：每个元素异或x
void xor_operation(int x) {
    if (x == 0) return;
    operation_list.push_back(-x);
    for (int i = 0; i < total_size; ++i) {
        data_arr[i] ^= x;
    }
    update_position_map();
}

// 计算辅助参数pa、pb
void calculate_params(int a_val, int b_val, int &pa, int &pb) {
    int delta = (b_val - a_val + total_size - block_size + total_size) % total_size;
    pa = pb = 0;
    for (int step = total_size / 2; step >= 2 * block_size; step /= 2) {
        if (delta >= step) {
            delta -= step;
            pb += step / 2;
        } else {
            pa += step / 2;
        }
    }
    pa += total_size / 2;
    pa += (a_val & (block_size - 1));
    pb += (a_val & (block_size - 1));
}

// 交换位置c和d的元素
void swap_elements(int c, int d) {
    if ((c / block_size) % 2 == (d / block_size) % 2) {
        int mid_pos;
        if ((c / block_size) % 2 == 0) {
            mid_pos = (c & (block_size - 1)) + block_size;
        } else {
            mid_pos = c & (block_size - 1);
        }
        swap_elements(c, mid_pos);
        swap_elements(d, mid_pos);
        swap_elements(c, mid_pos);
    } else {
        int pa, pb, pc, pd;
        calculate_params(target_a, target_b, pa, pb);
        calculate_params(c, d, pc, pd);

        add_operation((pc - c + total_size) % total_size);
        xor_operation(pc ^ pa);
        add_operation((target_a - pa + total_size) % total_size);
        magic_swap();
        add_operation((pa - target_a + total_size) % total_size);
        xor_operation(pc ^ pa);
        add_operation((c - pc + total_size) % total_size);
    }
}

// 排列结构体：处理块级操作
struct PermutationProcessor {
    int value_arr[MAX_ARRAY_SIZE];
    int arr_size;
    vector<int> sub_operations;

    // 递归求解块级操作序列
    bool process() {
        bool visited[100005] = {false};
        for (int i = 0; i < arr_size; ++i) {
            visited[i] = true;
        }
        for (int i = 0; i < arr_size; ++i) {
            if (!visited[i]) return false;
        }
        if (arr_size == 1) return true;

        PermutationProcessor left, right;
        left.arr_size = arr_size / 2;
        right.arr_size = arr_size / 2;

        for (int i = 0; i < arr_size / 2; ++i) {
            left.value_arr[i] = value_arr[i * 2] / 2;
            right.value_arr[i] = value_arr[i * 2 + 1] / 2;
        }

        if (!left.process() || !right.process()) return false;
        if (value_arr[0] % 2) {
            sub_operations.push_back(arr_size == 2 ? 1 : -1);
        }

        int temp_a = 0, temp_b = 0;
        for (int op : left.sub_operations) {
            if (op > 0) {
                sub_operations.push_back(-1);
                sub_operations.push_back(1);
            } else {
                sub_operations.push_back(op * 2);
                temp_a ^= (-op) * 2;
            }
        }
        if (temp_a) sub_operations.push_back(-temp_a);

        for (int op : right.sub_operations) {
            if (op > 0) {
                sub_operations.push_back(1);
                sub_operations.push_back(-1);
            } else {
                sub_operations.push_back(op * 2);
                temp_b ^= (-op) * 2;
            }
        }

        if ((temp_b & (arr_size / 2)) != (temp_a & (arr_size / 2))) {
            for (int i = 0; i < arr_size / 4; ++i) {
                sub_operations.push_back(-1);
                sub_operations.push_back(1);
            }
        }

        if (temp_a >= arr_size / 2) temp_a -= arr_size / 2;
        if (temp_b >= arr_size / 2) temp_b -= arr_size / 2;
        if (temp_a != temp_b) return false;

        vector<int> merged_ops;
        for (int op : sub_operations) {
            if (merged_ops.empty()) {
                merged_ops.push_back(op);
            } else {
                if (op < 0 && merged_ops.back() < 0) {
                    merged_ops.back() = -((-merged_ops.back()) ^ (-op));
                    if (merged_ops.back() == 0) merged_ops.pop_back();
                } else {
                    merged_ops.push_back(op);
                }
            }
        }
        swap(merged_ops, sub_operations);
        return true;
    }
};

int main() {
    // 输入：总大小、目标a/b、初始数组
    total_size = read_integer();
    target_a = read_integer();
    target_b = read_integer();
    for (int i = 0; i < total_size; ++i) {
        data_arr[i] = read_integer();
    }
    update_position_map();

    // 计算块大小：a和b的最大2的幂差
    block_size = (target_a - target_b + total_size) % total_size;
    block_size &= -block_size;
    if (block_size == 0) block_size = total_size;

    // 处理块级异或/加法操作
    if (block_size > 1) {
        PermutationProcessor processor;
        processor.arr_size = block_size;
        for (int i = 0; i < total_size; ++i) {
            processor.value_arr[i] = data_arr[i] & (block_size - 1);
        }
        if (!processor.process()) {
            printf("-1\n");
            return 0;
        }
        for (int op : processor.sub_operations) {
            if (op > 0) add_operation(op);
            else xor_operation(-op);
        }
    }

    // 逐块校验并修正元素位置
    for (int i = 0; i < block_size; ++i) {
        vector<int> block_elements;
        for (int j = i; j < total_size; j += block_size) {
            block_elements.push_back(data_arr[j]);
        }
        sort(block_elements.begin(), block_elements.end());

        bool is_valid = true;
        int ptr = 0;
        for (int j = i; j < total_size; j += block_size, ++ptr) {
            if (block_elements[ptr] != j) {
                is_valid = false;
                break;
            }
        }
        if (!is_valid) {
            printf("-1\n");
            return 0;
        }

        // 交换错位元素
        for (int j = i; j < total_size; j += block_size) {
            if (data_arr[j] != j) {
                swap_elements(j, data_arr[j]);
            }
        }
    }

    // 校验最终数组有序
    for (int i = 0; i < total_size; ++i) {
        assert(data_arr[i] == i);
    }

    // 输出操作序列
    printf("%d\n", (int)operation_list.size());
    for (int op : operation_list) {
        if (op == 0) {
            printf("0\n");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}

## B 长跑

In [1]:
import sys

def solve():
    # 读取所有标准输入并以空白字符分割
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    iterator = iter(input_data)
    
    while True:
        try:
            n_str = next(iterator)
        except StopIteration:
            break  # 所有测试用例读取完毕
        
        N = int(n_str)
        L = int(next(iterator))
        Maxn = int(next(iterator))
        S = int(next(iterator))
        
        stations = []
        for _ in range(N):
            p = int(next(iterator))
            c = int(next(iterator))
            stations.append((p, c))
            
        # 1. 休息 0 次的情况：初始体力足以直接跑到终点
        if Maxn >= L:
            print("Yes")
            continue
            
        # 预处理补给站：
        # - 去除位于终点或终点之后的补给站（因为到达那里已经算成功了）
        # - 去除单个花费就超过 S 的补给站
        # - 如果同一个位置有多个补给站，只保留最便宜的一个
        best_stations = {}
        for p, c in stations:
            if p >= L: 
                continue
            if c > S: 
                continue
            if p not in best_stations or c < best_stations[p]:
                best_stations[p] = c
                
        # 按照距离从左到右排序
        valid_stations = sorted(best_stations.items())
        n_valid = len(valid_stations)
        
        ans = "No"
        
        # 2. 休息 1 次的情况
        # 寻找是否存在一个补给站 p，满足：
        # - 能跑到该补给站: p <= Maxn
        # - 补给后能跑到终点: L - p <= Maxn
        # - 买得起: c <= S
        for p, c in valid_stations:
            if p <= Maxn and L - p <= Maxn and c <= S:
                ans = "Yes"
                break
        
        if ans == "Yes":
            print(ans)
            continue
            
        # 3. 休息 2 次的情况
        # 枚举第二个补给站 j 和第一个补给站 i (i < j)
        for j in range(n_valid):
            p2, c2 = valid_stations[j]
            # 第二个补给站必须能支撑到终点
            if L - p2 > Maxn:
                continue
                
            req_p1_min = p2 - Maxn
            req_c1_max = S - c2
            
            for i in range(j):
                p1, c1 = valid_stations[i]
                # 第一个补给站必须在初始体力范围内，如果超出了，后续的 p1 也不可能满足
                if p1 > Maxn:
                    break
                # 第一个补给站要能支撑到第二个补给站，且两个补给站的总花费不超过 S
                if p1 >= req_p1_min and c1 <= req_c1_max:
                    ans = "Yes"
                    break
            
            if ans == "Yes":
                break
                
        print(ans)

if __name__ == '__main__':
    solve()

## C 最长回文

In [ ]:
import sys

def solve():
    # 读取所有输入数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    N = int(input_data[0])
    A = input_data[1]
    B = input_data[2]
    
    # 字符串哈希预处理
    MOD = 10**18 + 9
    BASE = 131
    
    P = [1] * (N + 1)
    for i in range(1, N + 1):
        P[i] = (P[i-1] * BASE) % MOD
        
    def build_hash(s):
        h = [0] * (N + 1)
        for i in range(N):
            h[i+1] = (h[i] * BASE + ord(s[i])) % MOD
        return h

    A_R = A[::-1]
    hA = build_hash(A_R)
    hB = build_hash(B)
    
    # 马拉车算法寻找所有核心回文串
    def manacher(s):
        T = ['#'] * (2 * len(s) + 1)
        for i in range(len(s)):
            T[2 * i + 1] = s[i]
            
        P_arr = [0] * len(T)
        c = 0
        r = 0
        for i in range(len(T)):
            if i < r:
                P_arr[i] = min(r - i, P_arr[2 * c - i])
            while i - P_arr[i] - 1 >= 0 and i + P_arr[i] + 1 < len(T) and T[i - P_arr[i] - 1] == T[i + P_arr[i] + 1]:
                P_arr[i] += 1
            if i + P_arr[i] > r:
                c = i
                r = i + P_arr[i]
        return P_arr, T

    P_A, T_A = manacher(A)
    P_B, T_B = manacher(B)
    
    ans_max = 0
    
    # ================= Option A: 回文中心在 A 中 =================
    len_T_A = len(T_A)
    for i in range(len_T_A):
        l = (i - P_A[i]) >> 1
        r = ((i + P_A[i]) >> 1) - 1
        
        idxA = N - l
        idxB = r
        core_len = r - l + 1
        
        # 边界处理：如果无法向两侧延伸，仅靠核心长度能否更新答案
        if idxA < 0 or idxA >= N or idxB < 0 or idxB >= N:
            if core_len > ans_max:
                ans_max = core_len
            continue
            
        # 理论最大延伸长度
        max_len = N - idxA
        if N - idxB < max_len: 
            max_len = N - idxB
        
        # 🚨 终极优化：计算打破当前记录“至少需要”的延伸长度 req_k
        req_k = (ans_max - core_len) // 2 + 1
        if req_k <= 0:
            req_k = 0
        
        # 如果连理论最大值都达不到最低门槛，直接跳过
        if req_k > max_len:
            continue
            
        # O(1) 探路：直接看 req_k 长度的哈希值是否匹配
        if req_k > 0:
            ha = (hA[idxA + req_k] - hA[idxA] * P[req_k]) % MOD
            hb = (hB[idxB + req_k] - hB[idxB] * P[req_k]) % MOD
            if ha != hb:
                continue # 连门槛都没过，毫无悬念直接放弃！
                
        # 只有在确信能打破记录时，才进行二分查找寻找准确的延伸极值
        low = req_k
        high = max_len
        k = req_k
        while low <= high:
            mid = (low + high) >> 1
            ha = (hA[idxA + mid] - hA[idxA] * P[mid]) % MOD
            hb = (hB[idxB + mid] - hB[idxB] * P[mid]) % MOD
            if ha == hb:
                k = mid
                low = mid + 1
            else:
                high = mid - 1
                
        cand = core_len + (k << 1)
        if cand > ans_max:
            ans_max = cand

    # ================= Option B: 回文中心在 B 中 =================
    len_T_B = len(T_B)
    for i in range(len_T_B):
        l = (i - P_B[i]) >> 1
        r = ((i + P_B[i]) >> 1) - 1
        
        idxA = N - 1 - l
        idxB = r + 1
        core_len = r - l + 1
        
        if idxA < 0 or idxA >= N or idxB < 0 or idxB >= N:
            if core_len > ans_max:
                ans_max = core_len
            continue
            
        max_len = N - idxA
        if N - idxB < max_len: 
            max_len = N - idxB
            
        # 🚨 终极优化：同样的探路剪枝
        req_k = (ans_max - core_len) // 2 + 1
        if req_k <= 0:
            req_k = 0
        
        if req_k > max_len:
            continue
            
        if req_k > 0:
            ha = (hA[idxA + req_k] - hA[idxA] * P[req_k]) % MOD
            hb = (hB[idxB + req_k] - hB[idxB] * P[req_k]) % MOD
            if ha != hb:
                continue
                
        low = req_k
        high = max_len
        k = req_k
        while low <= high:
            mid = (low + high) >> 1
            ha = (hA[idxA + mid] - hA[idxA] * P[mid]) % MOD
            hb = (hB[idxB + mid] - hB[idxB] * P[mid]) % MOD
            if ha == hb:
                k = mid
                low = mid + 1
            else:
                high = mid - 1
                
        cand = core_len + (k << 1)
        if cand > ans_max:
            ans_max = cand

    print(ans_max)

if __name__ == '__main__':
    solve()

## D 优惠券

In [ ]:
#include <iostream>
#include <vector>
#include <string>

using namespace std;

// 应对极限数据，防止越界
const int MAX_X = 100005;
const int MAX_M = 500005;

int state[MAX_X] = {0};         // 0: 当前未拥有, 1: 当前正拥有
int last_buy[MAX_X] = {0};      // 记录 x 最近一次被购买(I)的行号
int last_unowned[MAX_X] = {0};  // 记录 x 最近一次被消耗(O)变为空白状态的行号
bool visited[MAX_X] = {false};  // 记录本轮是否操作过该编号

int bit[MAX_M] = {0}; // 树状数组，极低内存开销维护未使用的问号

// 树状数组：单点增加
void add(int i, int val, int m) {
    for (; i <= m; i += i & -i) {
        bit[i] += val;
    }
}

// 树状数组：查询前缀和
int query(int i) {
    int s = 0;
    for (; i > 0; i -= i & -i) {
        s += bit[i];
    }
    return s;
}

// 核心贪心：O(log M) 极速找出在 target 之后出现的第一个有效问号的位置
int find_first_greater(int target, int m_total) {
    int target_sum = query(target) + 1; 
    if (query(m_total) < target_sum) return -1; // 问号不够用了
    
    int pos = 0;
    int sum = 0;
    // 2^19 = 524288 > 500000，足以覆盖最大行数
    for (int i = 19; i >= 0; i--) { 
        if (pos + (1 << i) <= m_total && sum + bit[pos + (1 << i)] < target_sum) {
            pos += (1 << i);
            sum += bit[pos];
        }
    }
    return pos + 1; 
}

void solve() {
    int m;
    while (cin >> m) {
        vector<int> modified_x;
        int error_line = -1;
        
        for (int i = 1; i <= m; ++i) {
            string op;
            cin >> op;
            int x = 0;
            if (op == "I" || op == "O") {
                cin >> x;
            }
            
            if (error_line != -1) continue; // 报错后只读取输入，不处理逻辑
            
            // 兼容中英文字符，防止自测时输入法背锅
            if (op == "?" || op == "？") {
                add(i, 1, m); // 记录下行号 i 处有一个待命的问号
            } else if (op == "I") {
                if (!visited[x]) {
                    visited[x] = true;
                    modified_x.push_back(x);
                }
                
                if (state[x] == 0) {
                    state[x] = 1;
                    last_buy[x] = i; 
                } else {
                    // 当前已经拥有 x 却又来买，必须在上次购买之后找个问号把它"用掉"
                    int q_idx = find_first_greater(last_buy[x], m);
                    if (q_idx == -1) {
                        error_line = i;
                    } else {
                        add(q_idx, -1, m); // 消耗掉该问号
                        last_buy[x] = i;   // 刷新购买记录
                    }
                }
            } else if (op == "O") {
                if (!visited[x]) {
                    visited[x] = true;
                    modified_x.push_back(x);
                }
                
                if (state[x] == 1) {
                    state[x] = 0;
                    last_unowned[x] = i; 
                } else {
                    // 当前没有 x 却想使用，必须在上次失去之后找个问号把它"买下"
                    int q_idx = find_first_greater(last_unowned[x], m);
                    if (q_idx == -1) {
                        error_line = i;
                    } else {
                        add(q_idx, -1, m); 
                        last_unowned[x] = i; 
                    }
                }
            }
        }
        
        cout << error_line << "\n";
        
        // O(M) 清理数据，杜绝 memset 造成的运行超时
        for (int x : modified_x) {
            state[x] = 0;
            last_buy[x] = 0;
            last_unowned[x] = 0;
            visited[x] = false;
        }
        for (int i = 1; i <= m; ++i) {
            bit[i] = 0;
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    solve();
    return 0;
}

## E 任意点

In [ ]:
import sys

def solve():
    # 一次性读取所有输入，方便处理可能出现的多组测试数据或空白符问题
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    idx = 0
    n_tokens = len(input_data)
    
    while idx < n_tokens:
        n = int(input_data[idx])
        idx += 1
        
        points = []
        for _ in range(n):
            points.append((int(input_data[idx]), int(input_data[idx+1])))
            idx += 2
            
        # 1. 构建邻接表
        adj = [[] for _ in range(n)]
        for i in range(n):
            for j in range(i + 1, n):
                # 如果 x 坐标相同 或 y 坐标相同，说明它们可以直接互相到达，连一条边
                if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                    adj[i].append(j)
                    adj[j].append(i)
                    
        # 2. 遍历图，计算连通块的数量
        visited = [False] * n
        components = 0
        
        for i in range(n):
            if not visited[i]:
                components += 1
                # 使用栈进行 DFS 遍历，找出所有与 i 连通的点
                stack = [i]
                visited[i] = True
                while stack:
                    curr = stack.pop()
                    for neighbor in adj[curr]:
                        if not visited[neighbor]:
                            visited[neighbor] = True
                            stack.append(neighbor)
                            
        # 3. 最少添加的点数 = 连通块数量 - 1
        print(components - 1)

if __name__ == '__main__':
    solve()

## F 通配符匹配

In [ ]:
#include <iostream>
#include <string>
#include <vector>

using namespace std;

const int MAX_LEN = 100005;

// 计算 KMP 算法的 pi 数组 (部分匹配表)
vector<int> compute_pi(const string& P) {
    int m = P.length();
    vector<int> pi(m);
    for (int i = 1, j = 0; i < m; i++) {
        while (j > 0 && P[i] != P[j]) j = pi[j - 1];
        if (P[i] == P[j]) j++;
        pi[i] = j;
    }
    return pi;
}

// 快速找到普通子串 P 在目标串 S 中所有可能匹配的起始位置
void kmp_search(const string& S, const string& P, const vector<int>& pi, vector<char>& match, int N) {
    int m = P.length();
    for (int i = 0; i <= N; i++) match[i] = 0;
    
    // 如果分割出的子串为空字符串，那么它在 S 的任何位置都可以被视为匹配
    if (m == 0) {
        for (int i = 0; i <= N; i++) match[i] = 1;
        return;
    }
    for (int i = 0, j = 0; i < N; i++) {
        while (j > 0 && S[i] != P[j]) j = pi[j - 1];
        if (S[i] == P[j]) j++;
        if (j == m) {
            match[i - m + 1] = 1;
            j = pi[j - 1];
        }
    }
}

int main() {
    // 开启快读以应对大数据量
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    string pattern;
    if (!(cin >> pattern)) return 0;

    int n;
    cin >> n;

    // 1. 分割模式串，分离出普通的字符串片段以及通配符
    vector<string> parts;
    vector<char> wildcards;
    string current_part = "";
    for (char c : pattern) {
        if (c == '*' || c == '?') {
            parts.push_back(current_part);
            wildcards.push_back(c);
            current_part = "";
        } else {
            current_part += c;
        }
    }
    parts.push_back(current_part);

    int W = wildcards.size();
    vector<vector<int>> pis(W + 1);
    for (int i = 0; i <= W; i++) {
        pis[i] = compute_pi(parts[i]);
    }

    // 提前分配好空间，避免在循环内部重复分配内存，将内存开销降到最低
    // 使用 char 而不是 bool 来规避 vector<bool> 存在的底层优化代理带来的性能折损
    vector<vector<char>> matches(W + 1, vector<char>(MAX_LEN, 0));
    vector<char> dp(MAX_LEN, 0);
    vector<char> next_dp(MAX_LEN, 0);

    for (int k = 0; k < n; k++) {
        string S;
        cin >> S;
        int N = S.length();

        // 对于当前文件字符串 S，分别用 KMP 找出每一个普通字母块出现的所有合法位置
        for (int i = 0; i <= W; i++) {
            kmp_search(S, parts[i], pis[i], matches[i], N);
        }

        // 初始化 DP 数组
        for (int i = 0; i <= N; i++) dp[i] = 0;

        // 边界处理：初始块 parts[0] 必须要完美贴合 S 的最前缀
        int len0 = parts[0].length();
        if (len0 <= N && matches[0][0]) {
            dp[len0] = 1;
        }

        // 开始对通配符进行状态转移
        for (int i = 0; i < W; i++) {
            for (int j = 0; j <= N; j++) next_dp[j] = 0;
            int len_next = parts[i + 1].length();

            if (wildcards[i] == '?') {
                // 必须消耗且仅能消耗恰好一个字符
                for (int j = 0; j <= N; j++) {
                    if (dp[j]) {
                        int x = j + 1; // '?' 占一位
                        if (x + len_next <= N && matches[i + 1][x]) {
                            next_dp[x + len_next] = 1;
                        }
                    }
                }
            } else if (wildcards[i] == '*') {
                // `*` 匹配 0 到多个字符
                // 找到当前有效的最左侧匹配结束点 min_j
                int min_j = -1;
                for (int j = 0; j <= N; j++) {
                    if (dp[j]) {
                        min_j = j;
                        break;
                    }
                }
                
                if (min_j != -1) {
                    // 因为 `*` 包含任意长度，所以从 min_j 往后的任何合法出现位置都可以作为转移项
                    for (int x = min_j; x + len_next <= N; x++) {
                        if (matches[i + 1][x]) {
                            next_dp[x + len_next] = 1;
                        }
                    }
                }
            }
            // 滚动数组赋值
            for (int j = 0; j <= N; j++) dp[j] = next_dp[j];
        }

        // DP 最后的状态 dp[N] 记录了是否能够刚好消耗完字符串 S 的长度 N
        if (dp[N]) cout << "YES\n";
        else cout << "NO\n";
    }

    return 0;
}

## G 汉诺塔

In [ ]:
import sys

def solve():
    # 读取所有的输入并用空白符分割
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    
    # 解析优先级，值越小优先级越高 (0-5)
    priorities = {}
    for i in range(1, 7):
        if i < len(input_data):
            priorities[input_data[i]] = i - 1
            
    peg_names = ['A', 'B', 'C']
    
    # f[i][u] 表示将高度为 i 的塔从柱子 u 移走的步数
    # dest[i][u] 表示将高度为 i 的塔从柱子 u 移走后最终到达的柱子
    f = [[0] * 3 for _ in range(n + 1)]
    dest = [[0] * 3 for _ in range(n + 1)]
    
    # 基础情况：i = 1
    for u in range(3):
        # 找出除了 u 以外的另外两根柱子 v 和 w
        others = [x for x in range(3) if x != u]
        v = others[0]
        w = others[1]
        
        move1 = peg_names[u] + peg_names[v]
        move2 = peg_names[u] + peg_names[w]
        
        # 比较两个合法移动的优先级
        if priorities[move1] < priorities[move2]:
            dest[1][u] = v
        else:
            dest[1][u] = w
            
        f[1][u] = 1
        
    # 状态转移：i = 2 到 n
    for i in range(2, n + 1):
        for u in range(3):
            # 1. i-1 塔首先移动到的柱子 v
            v = dest[i-1][u]
            # 第三根空闲的柱子 w
            w = 3 - u - v 
            
            # 2. i-1 塔从 v 继续移动的目标
            nxt = dest[i-1][v]
            
            if nxt == w:
                # 情况1：i-1塔直接移动到大盘子上
                dest[i][u] = w
                f[i][u] = f[i-1][u] + 1 + f[i-1][v]
            else: 
                # nxt == u
                # 情况2：i-1塔退回原点，大盘子需要再移一次，i-1塔再追随一次
                dest[i][u] = v
                f[i][u] = f[i-1][u] + 1 + f[i-1][v] + 1 + f[i-1][u]
                
    # 所有盘子初始都在 A 柱子（索引 0）
    print(f[n][0])

if __name__ == '__main__':
    solve()

## H 马步距离

In [ ]:
import sys

def solve():
    # 读取所有标准输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    xp = int(input_data[0])
    yp = int(input_data[1])
    xs = int(input_data[2])
    ys = int(input_data[3])
    
    # 获取绝对坐标差
    dx = abs(xp - xs)
    dy = abs(yp - ys)
    
    # 保证 dx 是较大的那个，方便后续计算
    if dx < dy:
        dx, dy = dy, dx
        
    # 特判两个由于距离过近导致需要绕路的例外情况
    if dx == 1 and dy == 0:
        print(3)
        return
    if dx == 2 and dy == 2:
        print(4)
        return
        
    # 计算理论最短步数（利用向上取整的整数除法公式）
    # ⌈A / B⌉ 可以写成 (A + B - 1) // B
    ans = max((dx + 1) // 2, (dx + dy + 2) // 3)
    
    # 奇偶性修正：步数的奇偶性必须与 (dx + dy) 的奇偶性一致
    if ans % 2 != (dx + dy) % 2:
        ans += 1
        
    print(ans)

if __name__ == '__main__':
    solve()

## I 直方图最大矩形

In [ ]:
class Solution:
    def largestRectangleArea(self, heights: list[int]) -> int:
        # 在末尾添加一个高度为 0 的哨兵，强制弹出栈内所有剩余元素
        heights = heights + [0]
        # 栈内初始化一个 -1，作为虚拟的最左边界
        stack = [-1]
        max_area = 0
        
        for i in range(len(heights)):
            # 当遇到比栈顶元素矮的柱子时，说明栈顶柱子的右边界找到了
            while stack[-1] != -1 and heights[stack[-1]] >= heights[i]:
                # 弹出栈顶元素，将其高度作为矩形的高度
                h = heights[stack.pop()]
                # 此时新的栈顶就是左边界，i 就是右边界
                # 宽度 = 右边界 - 左边界 - 1
                w = i - stack[-1] - 1
                # 更新最大面积
                max_area = max(max_area, h * w)
            
            # 将当前元素的索引入栈，维持栈的单调递增
            stack.append(i)
            
        return max_area

if __name__ == "__main__":
    import sys
    import ast
    
    # 读取所有输入（处理可能的 ACM 模式）
    input_str = sys.stdin.read().strip()
    if input_str:
        try:
            # 将输入的字符串 "[3,4,7,8,1,2]" 解析为 Python 列表
            heights = ast.literal_eval(input_str)
            print(Solution().largestRectangleArea(heights))
        except Exception:
            pass

## J 消防局的设立

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    if n == 0:
        print(0)
        return
        
    parent = [0] * (n + 1)
    for i in range(2, n + 1):
        parent[i] = int(input_data[i - 1])
    
    # 子树中最远未覆盖节点到i的距离
    unc = [0] * (n + 1)
    # 子树中最近消防站到i的距离
    st = [float('inf')] * (n + 1)
    
    ans = 0
    
    # 自底向上遍历
    for i in range(n, 0, -1):
        # 已被覆盖
        if unc[i] + st[i] <= 2:
            unc[i] = -float('inf')
        
        # 必须建站
        if unc[i] == 2:
            ans += 1
            st[i] = 0
            unc[i] = -float('inf')
        
        # 向父节点传递信息
        if i > 1:
            p = parent[i]
            unc[p] = max(unc[p], unc[i] + 1)
            st[p] = min(st[p], st[i] + 1)
    
    # 根节点处理
    if unc[1] >= 0:
        ans += 1
    
    print(ans)

if __name__ == '__main__':
    solve()